# Climatron Optimal Model — T4 Training Pipeline

**Goal**: Train a bidirectional transformer encoder for climate fact-checking on Google Colab Free T4 GPU within 11 hours.

**Architecture**: v7-optimal — 8 layers, 384-d, GQA (6Q/2KV), SwiGLU, RMSNorm, RoPE, 25M params

**Pipeline**: Setup → Data → Pretrain (MLM) → Fine-tune (LoRA) → Evaluate → Report

**Budget**: ~3h actual workload, 11h allocated (8h margin for network/variance)

## Step 1: Boot the environment

Run `setup.sh` once at the start. It installs `uv`, sets up git, and writes the API values to `.env` for the rest of the notebook.

In [ ]:
import os
os.chdir("/content/ClimateClaimVerifier")
!bash setup.sh

from dotenv import load_dotenv

load_dotenv(".env")
GITHUB_PAT = os.getenv("GITHUB_PAT")
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")

print("✓ Environment ready")

## Step 2: Verify GPU Access

Check that we have a T4 GPU with enough VRAM. Colab Free typically provides T4 with ~15GB usable.

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA Capability: {props.major}.{props.minor}")
    # T4 is compute capability 7.5 — we can use FP16 but not BF16
    device = torch.device("cuda")
else:
    print("⚠ No GPU! Training will be extremely slow on CPU.")
    device = torch.device("cpu")

## Step 3: Define Model Architecture

All model components defined inline for self-contained Colab execution. This is a **bidirectional encoder** inspired by ModernBERT, optimized for data efficiency.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

# ─── Model Configuration ────────────────────────────────────────────
@dataclass
class ModelConfig:
    """Configuration for the optimal Climatron variant (v7-inspired)."""
    vocab_size: int = 32768
    d_model: int = 384           # Hidden dimension — small for data efficiency
    n_layers: int = 8            # 8 layers — enough depth for claim+evidence
    n_heads: int = 6             # Query heads
    n_kv_heads: int = 2          # KV heads (3:1 GQA ratio)
    d_intermediate: int = 1024   # FFN hidden: (8/3)*384 rounded to 256x
    max_seq_len: int = 1024      # Long enough for claim + 6 evidence passages
    dropout: float = 0.0         # No dropout in pretraining — MLM is regularization
    norm_type: str = "rmsnorm"
    attn_type: str = "gqa"
    ffn_type: str = "swiglu"
    pos_type: str = "rope"
    pad_token_id: int = 0
    bos_token_id: int = 2
    eos_token_id: int = 3
    mask_token_id: int = 32768  # Extended vocab for [MASK] during MLM
    bidirectional: bool = True
    pooling_type: str = "mean"  # Mean pool all non-pad tokens for classification
    tie_word_embeddings: bool = False  # False because vocab extended for [MASK]

config = ModelConfig()
print(f"Model: {config.n_layers}L/{config.d_model}d, GQA {config.n_heads}Q/{config.n_kv_heads}KV")
print(f"Position: RoPE, Norm: RMSNorm, FFN: SwiGLU, Bidirectional: True")

In [ ]:
# ─── Embedding Components ───────────────────────────────────────────

class TokenEmbedding(nn.Module):
    """Scaled token embedding: output = Embedding(x) × √d.
    The scaling factor follows the original Transformer paper to keep
    gradient magnitudes consistent between embedding and positional encoding."""
    def __init__(self, vocab_size, d_model, extended_size=None):
        super().__init__()
        actual_size = extended_size or vocab_size
        self.embedding = nn.Embedding(actual_size, d_model)
        self.scale = math.sqrt(d_model)
    def forward(self, x):
        return self.embedding(x) * self.scale

class RotaryPositionalEmbedding(nn.Module):
    """RoPE: Rotary Positional Embedding (Su et al., 2021).
    Applies rotation to Q and K vectors based on position. The dot product
    naturally encodes relative distance: q_m · k_n = f(m - n).
    This means attention patterns depend on relative, not absolute, positions."""
    def __init__(self, head_dim, max_seq_len=2048, theta=10000.0):
        super().__init__()
        self.head_dim = head_dim
        # Precompute complex frequencies: θ^(-2i/d) for each dimension pair
        freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, freqs)  # (max_seq_len, head_dim//2)
        self.register_buffer("freqs_cis", torch.polar(torch.ones_like(freqs), freqs))

    def apply_rotary(self, xq, xk, start_pos=0):
        """Apply rotation to query and key tensors."""
        S = xq.shape[2]
        freqs = self.freqs_cis[start_pos:start_pos+S].to(xq.device)
        # Reshape to complex, multiply, reshape back — this IS the rotation
        xq_reshape = xq.float().reshape(*xq.shape[:-1], -1, 2)
        xk_reshape = xk.float().reshape(*xk.shape[:-1], -1, 2)
        xq_complex = torch.view_as_complex(xq_reshape)
        xk_complex = torch.view_as_complex(xk_reshape)
        freqs = freqs.view([1]*(xq.ndim-2) + [S, self.head_dim//2])
        xq_rot = torch.view_as_real(xq_complex * freqs).flatten(-2)
        xk_rot = torch.view_as_real(xk_complex * freqs).flatten(-2)
        return xq_rot.type_as(xq), xk_rot.type_as(xk)

# ─── Normalization ───────────────────────────────────────────────────

class RMSNorm(nn.Module):
    """RMSNorm: x / RMS(x) × γ.
    Faster than LayerNorm — no mean subtraction, no bias term.
    Used in LLaMA, Mistral, ModernBERT."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight

In [ ]:
# ─── Grouped Query Attention ─────────────────────────────────────────

class GroupedQueryAttention(nn.Module):
    """GQA with RoPE: uses fewer KV heads than Q heads.
    Our config: 6 Q-heads, 2 KV-heads (3:1 ratio).
    KV heads are expanded via repeat_interleave to match Q heads.
    RoPE is applied to Q and K before KV expansion."""
    def __init__(self, config):
        super().__init__()
        d, h, kv = config.d_model, config.n_heads, config.n_kv_heads
        self.head_dim = d // h
        self.n_groups = h // kv
        self.dropout = config.dropout

        # Separate projections for Q, K, V, and output
        self.W_q = nn.Linear(d, h * self.head_dim)
        self.W_k = nn.Linear(d, kv * self.head_dim)
        self.W_v = nn.Linear(d, kv * self.head_dim)
        self.W_o = nn.Linear(d, d)
        self.rope = RotaryPositionalEmbedding(self.head_dim, config.max_seq_len)

    def forward(self, x, causal=False):
        B, S, D = x.shape
        q = self.W_q(x).view(B, S, -1, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, S, -1, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, S, -1, self.head_dim).transpose(1, 2)

        # Apply RoPE: rotate Q and K by position-dependent complex numbers
        q, k = self.rope.apply_rotary(q, k)

        # Expand KV heads to match Q heads (repeat each KV head for its group)
        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)

        # Flash Attention via PyTorch's fused SDPA kernel
        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0)
        return self.W_o(out.transpose(1,2).contiguous().view(B,S,D))

# ─── SwiGLU Feed-Forward ─────────────────────────────────────────────

class SwiGLUFFN(nn.Module):
    """SwiGLU: SiLU(gate(x)) ⊙ up(x), then down projection.
    The gate controls information flow — SiLU is a smooth sigmoid-like gate.
    Element-wise multiplication with 'up' creates a learned filter."""
    def __init__(self, d_model, d_intermediate, dropout=0.0):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_intermediate)
        self.up_proj = nn.Linear(d_model, d_intermediate)
        self.down_proj = nn.Linear(d_intermediate, d_model)
        self.dropout = dropout

    def forward(self, x):
        gate = F.silu(self.gate_proj(x))  # SiLU activation on gate path
        up = self.up_proj(x)               # Linear transformation on up path
        out = gate * up                    # Element-wise gating
        if self.dropout > 0 and self.training:
            out = F.dropout(out, p=self.dropout)
        return self.down_proj(out)

In [ ]:
# ─── Transformer Block & Full Model ──────────────────────────────────

class TransformerBlock(nn.Module):
    """Pre-norm Transformer block: Attention + FFN with residual connections.
    x = x + Attn(Norm1(x)),  x = x + FFN(Norm2(x))"""
    def __init__(self, config):
        super().__init__()
        self.norm1 = RMSNorm(config.d_model)
        self.norm2 = RMSNorm(config.d_model)
        self.attn = GroupedQueryAttention(config)
        self.ffn = SwiGLUFFN(config.d_model, config.d_intermediate, config.dropout)

    def forward(self, x, causal=False):
        x = x + self.attn(self.norm1(x), causal=causal)
        x = x + self.ffn(self.norm2(x))
        return x


class ClimatronForPretraining(nn.Module):
    """Bidirectional encoder for MLM pretraining.
    Processes the full sequence with bidirectional attention, then predicts
    masked tokens via an LM head."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        extended_vocab = max(config.vocab_size, config.mask_token_id + 1)
        self.token_embedding = TokenEmbedding(config.vocab_size, config.d_model, extended_vocab)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.final_norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, extended_vocab)

    def forward(self, input_ids, attention_mask=None):
        x = self.token_embedding(input_ids)
        causal = not self.config.bidirectional  # False = full bidirectional attention
        for block in self.blocks:
            x = block(x, causal=causal)
        x = self.final_norm(x)
        return self.lm_head(x), {}  # logits, aux_losses (empty for non-MoE)


class ClimatronForClassification(nn.Module):
    """Classification head on top of pretrained encoder.
    Uses mean pooling over all non-pad token outputs (ModernBERT-style).
    Input format: <bos> claim <sep> ev1 <sep> ev2 ... <eos>"""
    def __init__(self, config, pretrained_model):
        super().__init__()
        self.config = config
        self.token_embedding = pretrained_model.token_embedding
        self.blocks = pretrained_model.blocks
        self.final_norm = pretrained_model.final_norm
        self.classification_head = nn.Linear(config.d_model, 4)

    def forward(self, input_ids, attention_mask=None):
        x = self.token_embedding(input_ids)
        for block in self.blocks:
            x = block(x, causal=False)
        x = self.final_norm(x)
        # Mean pooling: average over all non-pad positions
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)
        return self.classification_head(x)

# Quick test: instantiate model and check parameter count
model = ClimatronForPretraining(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model created: {n_params:,} parameters ({n_params/1e6:.1f}M)")
print(f"  VRAM (fp32): ~{n_params*4/1e9:.1f} GB,  (fp16): ~{n_params*2/1e9:.1f} GB")

## Step 4: Data Loading

Stream from HuggingFace datasets to avoid downloading 10BT of data. We use the hf-mirror.com endpoint for faster access from mainland China.

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"  # Faster for mainland China

import sentencepiece as spm
from datasets import load_dataset
from torch.utils.data import DataLoader
import random

# ─── Tokenizer (SentencePiece BPE) ────────────────────────────────────
class SimpleTokenizer:
    """Lightweight SentencePiece BPE tokenizer wrapper.
    If a pretrained model file doesn't exist, we train a tiny one on the fly.
    For production, upload a proper 32K-vocab SP model."""
    PAD, UNK, BOS, EOS, SEP = 0, 1, 2, 3, 4

    def __init__(self, model_path=None):
        if model_path and os.path.exists(model_path):
            self.sp = spm.SentencePieceProcessor(model_file=model_path)
            self.vocab_size = self.sp.vocab_size()
        else:
            # Fallback: train a tiny BPE tokenizer on sample text
            self._train_tiny_tokenizer()
        self.pad_token_id = self.PAD
        self.bos_token_id = self.BOS
        self.eos_token_id = self.EOS

    def _train_tiny_tokenizer(self):
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
            f.write("Climate change global warming IPCC report evidence claim supports refutes disputed. " * 1000)
            corpus = f.name
        spm.SentencePieceTrainer.train(
            input=corpus, model_prefix='/tmp/sp', vocab_size=8000,
            model_type='bpe', pad_id=0, unk_id=1, bos_id=2, eos_id=3,
            user_defined_symbols=['<sep>'])
        self.sp = spm.SentencePieceProcessor(model_file='/tmp/sp.model')
        self.vocab_size = self.sp.vocab_size()
        os.unlink(corpus)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = self.sp.encode(text, out_type=int)
        if add_bos: ids = [self.BOS] + ids
        if add_eos: ids = ids + [self.EOS]
        return ids

tokenizer = SimpleTokenizer()
print(f"✓ Tokenizer ready: vocab_size={tokenizer.vocab_size}")

# ─── Streaming Dataset ───────────────────────────────────────────────
class StreamingPretrainDataset(torch.utils.data.IterableDataset):
    """Streaming dataset over HuggingFace FineWeb-Edu sample-10BT.
    Yields tokenized sequences of exactly max_seq_len tokens.
    Uses the datasets streaming API — no disk cache needed."""
    def __init__(self, tokenizer, max_seq_len=1024):
        self.ds = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT",
                               streaming=True, split="train")
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __iter__(self):
        for example in self.ds:
            tokens = self.tokenizer.encode(example["text"], add_bos=True, add_eos=True)
            yield tokens[:self.max_seq_len]
    def __len__(self):
        return 10_000_000  # approximate for progress bars

# ─── MLM Collator ────────────────────────────────────────────────────
class MLMCollator:
    """BERT-style MLM collator: replaces 15% of tokens with [MASK], random, or unchanged.
    Labels contain original tokens at masked positions, -100 elsewhere."""
    def __init__(self, pad_id, mask_id, bos_id=2, eos_id=3, max_len=1024, mlm_prob=0.15):
        self.pad_id, self.mask_id = pad_id, mask_id
        self.bos_id, self.eos_id = bos_id, eos_id
        self.max_len, self.mlm_prob = max_len, mlm_prob

    def __call__(self, batch):
        batch_max = min(max(len(s) for s in batch), self.max_len)
        B = len(batch)
        input_ids = torch.full((B, batch_max), self.pad_id, dtype=torch.long)
        labels = torch.full((B, batch_max), -100, dtype=torch.long)
        attention_mask = torch.zeros((B, batch_max), dtype=torch.long)

        for i, seq in enumerate(batch):
            n = min(len(seq), batch_max)
            seq_t = torch.tensor(seq[:n], dtype=torch.long)
            # Only mask non-special tokens
            special = {self.pad_id, self.bos_id, self.eos_id, self.mask_id}
            eligible = torch.tensor([t not in special for t in seq_t.tolist()], dtype=torch.bool)
            if eligible.sum() == 0:
                input_ids[i,:n] = seq_t; attention_mask[i,:n] = 1; continue

            n_mask = max(1, int(eligible.sum().item() * self.mlm_prob))
            indices = eligible.nonzero(as_tuple=False).squeeze(-1)
            mask_idx = indices[torch.randperm(len(indices))[:n_mask]]

            masked = seq_t.clone()
            labels[i, mask_idx] = seq_t[mask_idx]
            rng = torch.rand(n_mask)
            masked[mask_idx[rng < 0.8]] = self.mask_id
            replace = (rng >= 0.8) & (rng < 0.9)
            if replace.any():
                random_toks = torch.randint(5, min(seq_t.max().item()+1, 8000), (replace.sum().item(),))
                masked[mask_idx[replace]] = random_toks
            input_ids[i,:n] = masked; attention_mask[i,:n] = 1
        return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

print("✓ Data pipeline ready")

## Step 5: Pretraining (Masked Language Modeling)

Train the bidirectional encoder on 200M tokens of educational web text. The model learns to predict masked words from context in both directions — building deep semantic understanding.

In [ ]:
from tqdm import tqdm
import time
from datetime import timedelta

# ─── Training Config ─────────────────────────────────────────────────
BATCH_SIZE = 16           # Fits T4 16GB at seq_len=1024
GRAD_ACCUM = 1            # Single-step accumulation
SEQ_LEN = 1024
NUM_TOKENS = 200_000_000  # 200M tokens — ~2h on T4
LR = 3e-4
WARMUP_FRAC = 0.05
STABLE_FRAC = 0.85

dataset = StreamingPretrainDataset(tokenizer, max_seq_len=SEQ_LEN)
collator = MLMCollator(tokenizer.pad_token_id, config.mask_token_id,
                       tokenizer.bos_token_id, tokenizer.eos_token_id, SEQ_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=collator)

model = ClimatronForPretraining(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=0.1)

# WSD Scheduler: 5% warmup → 85% stable → 10% linear decay
tokens_per_step = BATCH_SIZE * GRAD_ACCUM * SEQ_LEN
total_steps = NUM_TOKENS // tokens_per_step
warmup_steps = max(1, int(total_steps * WARMUP_FRAC))

def get_lr(step):
    if step < warmup_steps:
        return LR * step / warmup_steps
    if step < total_steps * (WARMUP_FRAC + STABLE_FRAC):
        return LR
    progress = (step - total_steps * (WARMUP_FRAC + STABLE_FRAC)) / (total_steps * 0.10)
    return LR * (1.0 - progress)

scaler = torch.amp.GradScaler('cuda', enabled=True)  # FP16 mixed precision
tokens_seen = 0
step = 0
running_loss = 0.0
start_time = time.time()

print(f"Pretraining: {NUM_TOKENS:,} tokens, batch={BATCH_SIZE}, seq={SEQ_LEN}")
print(f"Steps: {total_steps:,}, warmup: {warmup_steps}, LR: {LR:.0e}")

for batch in loader:
    model.train()
    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)

    with torch.amp.autocast('cuda', enabled=True):
        logits, _ = model(input_ids)
        # MLM loss: only on masked positions (labels != -100)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()

    # LR schedule
    for pg in optimizer.param_groups:
        pg['lr'] = get_lr(step)

    running_loss += loss.item()
    tokens_seen += tokens_per_step
    step += 1

    if step % 100 == 0:
        elapsed = time.time() - start_time
        speed = tokens_seen / elapsed if elapsed > 0 else 0
        pct = 100 * tokens_seen / NUM_TOKENS
        eta = (NUM_TOKENS - tokens_seen) / speed if speed > 0 else 0
        print(f"  Step {step:>6d} | {pct:5.1f}% | loss={running_loss/100:.4f} | "
              f"speed={speed:,.0f} tok/s | ETA={timedelta(seconds=int(eta))}")
        running_loss = 0.0

    if step % 2000 == 0:
        torch.save(model.state_dict(), f"pretrain_step{step}.pt")

    if tokens_seen >= NUM_TOKENS:
        break

# Save final pretrained model
torch.save(model.state_dict(), "pretrain_final.pt")
total_time = time.time() - start_time
print(f"\n✓ Pretraining complete: {tokens_seen:,} tokens in {timedelta(seconds=int(total_time))}")
print(f"  Avg speed: {tokens_seen/total_time:,.0f} tok/s")
print(f"  Model saved to pretrain_final.pt")

## Step 6: Fine-Tuning with LoRA

Apply Low-Rank Adaptation (LoRA) to the pretrained encoder, then train for 4-way classification with LDAM + Class-Balanced + DRW loss.

In [ ]:
# ─── LoRA Module ─────────────────────────────────────────────────────
class LoRALinear(nn.Module):
    """Low-Rank Adaptation: adds trainable A@B update to frozen Linear layer.
    A ∈ R^(d×r), B ∈ R^(r×d), r << d. Only A and B are trained."""
    def __init__(self, base, r=8, alpha=16, dropout=0.1):
        super().__init__()
        self.base = base
        for p in base.parameters(): p.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(base.in_features, r, device=base.weight.device, dtype=base.weight.dtype))
        self.B = nn.Parameter(torch.zeros(r, base.out_features, device=base.weight.device, dtype=base.weight.dtype))
        nn.init.kaiming_uniform_(self.A, a=5**0.5)
        self.scale = alpha / r
        self.lora_dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_dropout(x) @ self.A @ self.B

def apply_lora(model, r=8, alpha=16, dropout=0.1):
    """Recursively replace target Linear layers with LoRALinear wrappers."""
    target_names = {"W_q", "W_k", "W_v", "W_o", "gate_proj", "up_proj", "down_proj"}
    for name, child in model.named_children():
        if isinstance(child, nn.Linear) and name in target_names:
            setattr(model, name, LoRALinear(child, r, alpha, dropout))
        else:
            apply_lora(child, r, alpha, dropout)
    return model

# Load pretrained model and create classifier
model = ClimatronForPretraining(config).to(device)
model.load_state_dict(torch.load("pretrain_final.pt", map_location=device))
classifier = ClimatronForClassification(config, model).to(device)
classifier = apply_lora(classifier, r=8, alpha=16, dropout=0.1)

lora_params = [p for n, p in classifier.named_parameters() if p.requires_grad]
print(f"LoRA trainable: {sum(p.numel() for p in lora_params):,} / "
      f"{sum(p.numel() for p in classifier.parameters()):,} params")

In [ ]:
# ─── Class Imbalance Loss (LDAM + Class-Balanced + DRW) ──────────────

class ImbalancedLoss(nn.Module):
    """Combined loss: LDAM margins + Class-Balanced weights + Deferred Re-weighting.
    - LDAM: larger decision margins for minority classes (margin ∝ 1/√n)
    - CB: re-weight by effective number of samples (accounts for diminishing returns)
    - DRW: train epoch 1 without weights, enable weights from epoch 2 onward"""
    def __init__(self, class_counts, ldam_margin=0.3, cb_beta=0.999, use_drw=True):
        super().__init__()
        self.margins = ldam_margin / (class_counts.float() ** 0.25)
        effective = (1 - cb_beta ** class_counts) / (1 - cb_beta)
        self.cb_weights = effective.sum() / (len(class_counts) * effective)
        self.use_drw = use_drw
        self.reweight_enabled = not use_drw

    def enable_reweight(self):
        self.reweight_enabled = True

    def forward(self, logits, targets):
        # LDAM: subtract margin from true-class logit
        batch_margins = self.margins.to(logits.device)[targets]
        logits_adj = logits.clone()
        logits_adj.scatter_add_(1, targets.unsqueeze(1), -batch_margins.unsqueeze(1))
        # Optional CB weight
        weight = self.cb_weights.to(logits.device) if self.reweight_enabled else None
        return F.cross_entropy(logits_adj, targets, weight=weight)

# Compute class counts from training data (use known distribution)
class_counts = torch.tensor([519., 199., 386., 124.], device=device)
loss_fn = ImbalancedLoss(class_counts, use_drw=True)

In [ ]:
# ─── Fine-Tuning Loop ────────────────────────────────────────────────
# For Colab, we use a small pre-loaded dataset (or inline data)
# In practice, upload train-claims.json and evidence.json to Colab

import json

# Load training data — adjust paths for your Colab environment
try:
    with open("data/train-claims.json") as f: train_claims = json.load(f)
    with open("data/dev-claims.json") as f: dev_claims = json.load(f)
    with open("data/evidence.json") as f: evidence = json.load(f)
    print(f"Loaded: {len(train_claims)} train, {len(dev_claims)} dev, {len(evidence)} evidence")
except FileNotFoundError:
    print("⚠ Data files not found. Upload train-claims.json, dev-claims.json, and evidence.json to Colab.")
    print("  Using synthetic data for demonstration.")
    train_claims = {f"c{i}": {"claim_text": f"Claim {i}", "claim_label": ["SUPPORTS","REFUTES","NOT_ENOUGH_INFO","DISPUTED"][i%4], "evidences": ["ev1","ev2"]} for i in range(100)}
    dev_claims = {f"c{i}": {"claim_text": f"Dev claim {i}", "claim_label": "SUPPORTS", "evidences": ["ev1"]} for i in range(20)}
    evidence = {"ev1": "Climate change evidence passage.", "ev2": "IPCC report finding."}

from torch.utils.data import DataLoader, Dataset

class ClaimDataset(Dataset):
    def __init__(self, claims):
        self.items = list(claims.items())
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        cid, data = self.items[idx]
        return {"claim_id": cid, "claim_text": data["claim_text"],
                "claim_label": data.get("claim_label"), "evidences": data.get("evidences", [])}

class ClassificationCollator:
    """Formats claim + evidence into <bos> claim <sep> ev1 <sep> ... <eos>."""
    def __init__(self, tokenizer, evidence_lookup, max_len=512, max_ev=6):
        self.tokenizer = tokenizer
        self.ev = evidence_lookup
        self.max_len = max_len
        self.max_ev = max_ev
        label_map = {"SUPPORTS": 0, "REFUTES": 1, "NOT_ENOUGH_INFO": 2, "DISPUTED": 3}
        self.label_map = label_map

    def __call__(self, batch):
        all_ids, labels = [], []
        for item in batch:
            ev_texts = [self.ev.get(eid, "") for eid in item["evidences"][:self.max_ev]]
            parts = [item["claim_text"]] + ev_texts
            formatted = "<bos>" + "<sep>".join(parts) + "<eos>"
            ids = self.tokenizer.encode(formatted)
            ids = ids[:self.max_len-1] + [self.tokenizer.eos_token_id]
            all_ids.append(ids)
            labels.append(self.label_map.get(item["claim_label"], -1))
        B = len(batch); M = min(max(len(s) for s in all_ids), self.max_len)
        input_ids = torch.full((B, M), tokenizer.pad_token_id, dtype=torch.long)
        mask = torch.zeros((B, M), dtype=torch.long)
        for i, s in enumerate(all_ids):
            n = min(len(s), M); input_ids[i,:n] = torch.tensor(s[:n]); mask[i,:n] = 1
        return {"input_ids": input_ids, "attention_mask": mask, "labels": torch.tensor(labels)}

train_ds = ClaimDataset(train_claims)
dev_ds = ClaimDataset(dev_claims)
collator = ClassificationCollator(tokenizer, evidence, max_len=512)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collator)
dev_loader = DataLoader(dev_ds, batch_size=4, shuffle=False, collate_fn=collator)

# Training
optimizer = torch.optim.AdamW(lora_params, lr=2e-4, weight_decay=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=True)
best_acc = 0; best_state = None

for epoch in range(3):
    if epoch == 1:  # DRW: enable class re-weighting from epoch 2
        loss_fn.enable_reweight()
        print("  DRW: class re-weighting enabled")

    classifier.train()
    total_loss = 0
    for batch in train_loader:
        ids, mask, targets = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
        with torch.amp.autocast('cuda', enabled=True):
            logits = classifier(ids, mask)
            loss = loss_fn(logits, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        total_loss += loss.item()

    # Validation
    classifier.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in dev_loader:
            ids, mask, targets = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
            preds = classifier(ids, mask).argmax(-1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    acc = correct / total if total > 0 else 0
    print(f"Epoch {epoch+1}: train_loss={total_loss/len(train_loader):.4f} val_acc={acc:.4f}")
    if acc > best_acc:
        best_acc = acc; best_state = {k: v.cpu().clone() for k, v in classifier.state_dict().items()}

if best_state:
    classifier.load_state_dict(best_state)
torch.save(classifier.state_dict(), "classifier_final.pt")
print(f"\n✓ Fine-tuning complete. Best val_acc: {best_acc:.4f}")

## Step 7: Evaluation & Report Generation

Evaluate the fine-tuned classifier on the dev set and generate a comparison report.

In [ ]:
from collections import Counter

# Evaluate on dev set
classifier.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in dev_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        targets = batch["labels"].to(device)
        logits = classifier(ids, mask)
        preds = logits.argmax(-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(targets.cpu().tolist())

# Per-class accuracy
labels_names = ["SUPPORTS", "REFUTES", "NOT_ENOUGH", "DISPUTED"]
print("\n" + "="*60)
print("  FINAL EVALUATION")
print("="*60)
for i, name in enumerate(labels_names):
    total = sum(1 for l in all_labels if l == i)
    correct = sum(1 for p, l in zip(all_preds, all_labels) if p == i and l == i)
    print(f"  {name:<15}: {correct}/{total} ({100*correct/total:.1f}%)" if total > 0 else f"  {name}: N/A")

overall = sum(1 for p, l in zip(all_preds, all_labels) if p == l) / max(len(all_labels), 1)
print(f"\n  OVERALL ACCURACY: {overall:.4f}")

# Generate report
report = f"""# Climatron Optimal Model — T4 Training Report

## Model
- Architecture: {config.n_layers}L/{config.d_model}d, GQA {config.n_heads}Q/{config.n_kv_heads}KV
- Parameters: {sum(p.numel() for p in classifier.parameters()):,}
- Pretraining: MLM on {NUM_TOKENS:,} tokens FineWeb-Edu
- Fine-tuning: LoRA (r=8) + LDAM-DRW + Class-Balanced

## Results
- Overall Accuracy: {overall:.4f}
- Classes: SUPPORTS, REFUTES, NOT_ENOUGH, DISPUTED
"""

with open("training_report.md", "w") as f:
    f.write(report)
print(f"\n✓ Report saved to training_report.md")
print(f"✓ All done! Model files: pretrain_final.pt, classifier_final.pt")